# Toy ANN2SNN BrainScaleS-2 hardware-in-the-loop

Use the `EBRAINS-experimental` kernel. This notebook only configures the hardware client, invokes the repository CLI, and displays artifacts; experiment logic remains in `brainscales2_toy_hil.py`.

In [ ]:
%pip install --quiet --disable-pip-version-check jaxtyping matplotlib

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import sys

start = Path.cwd().resolve()
repo_root = next((p for p in (start, *start.parents) if (p / 'scripts/evaluation/brainscales2_toy_hil.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Could not locate the delayed-temporal repository root')
os.chdir(repo_root)
CLI = repo_root / 'scripts/evaluation/brainscales2_toy_hil.py'
RUN_TRAIN = False
RUN_LOCAL_REPLAY = False
RUN_HAGEN_PROBE = False
RUN_HARDWARE_SMOKE = False
RUN_YINYANG_FULL = False
RUN_MNIST_BENCHMARK = False
HAGEN_CALIBRATION_PATH = None  # None downloads the current chip's nightly calibration.
SPIKING_CALIBRATION_PATH = None  # Explicit Path overrides the run-local download.
HAGEN_HIDDEN_SHIFT = 1  # Replace with hagen_probe.json recommendation.
MNIST_HARDWARE_SAMPLES = 128  # Inspect runtime.json before increasing this.
run_label = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
artifact_root = repo_root / 'artifacts/brainscales2-toy' / run_label
artifact_root.mkdir(parents=True, exist_ok=True)
checkpoint_dir = artifact_root / 'checkpoint'
print('repository:', repo_root)
print('python:', sys.executable)
print('artifacts:', artifact_root)

In [ ]:
import inspect
import torch
import hxtorch
import hxtorch.perceptron
import hxtorch.spiking as hxsnn
print('Python:', sys.version)
print('torch:', torch.__version__)
print('hxtorch:', getattr(hxtorch, '__version__', 'unknown'))
print('Perceptron Linear:', inspect.signature(hxtorch.perceptron.nn.Linear))
print('Experiment:', inspect.signature(hxsnn.Experiment))
print('LIF:', inspect.signature(hxsnn.LIF))

## Configure the shared hardware client

The official helper checkout lives under `/tmp`, not the read-only or shared project parent. After allocating the current chip, missing Hagen and spiking calibration paths are downloaded into the run directory; explicit paths remain supported as overrides.

In [ ]:
hardware_requested = RUN_HAGEN_PROBE or RUN_HARDWARE_SMOKE or RUN_YINYANG_FULL or RUN_MNIST_BENCHMARK
if hardware_requested:
    demos_root = Path('/tmp/brainscales2-demos')
    if not demos_root.is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'jupyter-notebooks-experimental', 'https://github.com/electronicvisions/brainscales2-demos.git', str(demos_root)], check=True)
    sys.path.insert(0, str(demos_root))
    from _static.common.helpers import save_nightly_calibration, setup_hardware_client
    setup_hardware_client()
    calibration_dir = artifact_root / 'calibration'
    calibration_dir.mkdir(parents=True, exist_ok=True)
    if HAGEN_CALIBRATION_PATH is None:
        HAGEN_CALIBRATION_PATH = calibration_dir / 'hagen_cocolist.pbin'
        if not HAGEN_CALIBRATION_PATH.is_file():
            save_nightly_calibration(HAGEN_CALIBRATION_PATH.name, folder=str(calibration_dir))
    else:
        HAGEN_CALIBRATION_PATH = Path(HAGEN_CALIBRATION_PATH).expanduser().resolve()
    if SPIKING_CALIBRATION_PATH is None:
        SPIKING_CALIBRATION_PATH = calibration_dir / 'spiking_cocolist.pbin'
        if not SPIKING_CALIBRATION_PATH.is_file():
            save_nightly_calibration(SPIKING_CALIBRATION_PATH.name, folder=str(calibration_dir))
    else:
        SPIKING_CALIBRATION_PATH = Path(SPIKING_CALIBRATION_PATH).expanduser().resolve()
    for calibration_path in (HAGEN_CALIBRATION_PATH, SPIKING_CALIBRATION_PATH):
        if not calibration_path.is_file():
            raise FileNotFoundError(calibration_path)
    print('hardware client configured')
    print('Hagen calibration:', HAGEN_CALIBRATION_PATH)
    print('spiking calibration:', SPIKING_CALIBRATION_PATH)

In [ ]:
def run_cli(*arguments):
    command = [sys.executable, str(CLI), *(str(value) for value in arguments)]
    print(' '.join(command))
    subprocess.run(command, cwd=repo_root, check=True)

def calibration_args():
    paths = (HAGEN_CALIBRATION_PATH, SPIKING_CALIBRATION_PATH)
    if any(path is None for path in paths):
        raise RuntimeError('Run the hardware-client cell to resolve both calibration files')
    resolved = tuple(Path(path).expanduser().resolve() for path in paths)
    missing = [path for path in resolved if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'Missing calibration files: {missing}')
    return ['--hagen-calibration', resolved[0], '--spiking-calibration', resolved[1]]

In [ ]:
if RUN_TRAIN:
    run_cli('--phase', 'train', '--task', 'yinyang', '--architecture', 'yy-30', '--output-dir', checkpoint_dir)
    run_cli('--phase', 'convert', '--task', 'yinyang', '--architecture', 'yy-30', '--checkpoint', checkpoint_dir / 'checkpoint.pt', '--output-dir', checkpoint_dir)

In [ ]:
if RUN_LOCAL_REPLAY:
    run_cli('--phase', 'local-eval', '--task', 'yinyang', '--architecture', 'yy-30', '--checkpoint', checkpoint_dir / 'checkpoint.pt', '--converted-checkpoint', checkpoint_dir / 'converted_checkpoint.pt', '--pwm-backend', 'torch', '--pool-backend', 'replay', '--output-dir', artifact_root / 'replay')

In [ ]:
if RUN_HAGEN_PROBE:
    run_cli('--phase', 'probe-hagen', '--task', 'yinyang', '--architecture', 'yy-30', '--checkpoint', checkpoint_dir / 'checkpoint.pt', '--converted-checkpoint', checkpoint_dir / 'converted_checkpoint.pt', '--pwm-backend', 'hagen-hardware', '--hagen-hidden-shift', HAGEN_HIDDEN_SHIFT, '--output-dir', artifact_root / 'hagen_probe', *calibration_args())

In [ ]:
common_hardware = ['--checkpoint', checkpoint_dir / 'checkpoint.pt', '--converted-checkpoint', checkpoint_dir / 'converted_checkpoint.pt', '--pwm-backend', 'hagen-hardware', '--pool-backend', 'hardware', '--hagen-hidden-shift', HAGEN_HIDDEN_SHIFT, *calibration_args()] if (RUN_HARDWARE_SMOKE or RUN_YINYANG_FULL or RUN_MNIST_BENCHMARK) else []
if RUN_HARDWARE_SMOKE:
    run_cli('--phase', 'hardware-smoke', '--task', 'yinyang', '--architecture', 'yy-30', '--quick', '--output-dir', artifact_root / 'hardware_smoke', *common_hardware)
if RUN_YINYANG_FULL:
    run_cli('--phase', 'hardware-eval', '--task', 'yinyang', '--architecture', 'yy-30', '--pool-sizes', 1, 2, 4, 8, 16, '--output-dir', artifact_root / 'yinyang_full', *common_hardware)

In [ ]:
if RUN_MNIST_BENCHMARK:
    mnist_checkpoint = artifact_root / 'mnist_checkpoint'
    run_cli('--phase', 'hardware-eval', '--task', 'mnist', '--architecture', 'mnist-30', '--checkpoint', mnist_checkpoint / 'checkpoint.pt', '--converted-checkpoint', mnist_checkpoint / 'converted_checkpoint.pt', '--max-test-samples', MNIST_HARDWARE_SAMPLES, '--output-dir', artifact_root / 'mnist_benchmark', *common_hardware[4:])

In [ ]:
for manifest_path in sorted(artifact_root.rglob('manifest.json')):
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print(manifest_path.relative_to(artifact_root), {'task': manifest.get('task'), 'test_samples': manifest.get('test_samples'), 'conditions': len(manifest.get('conditions', []))})
for runtime_path in sorted(artifact_root.rglob('runtime.json')):
    print(runtime_path.relative_to(artifact_root), json.loads(runtime_path.read_text(encoding='utf-8')).get('mnist_estimates'))